# Integrasi Chatbot Sejarah Indonesia

Uji pemrosesan pertanyaan dan pencarian kalimat menggunakan BM25.

## 1. Persiapan modul dan korpus

In [1]:
import csv
import json
from pathlib import Path

from question_processing import process_question
from passage_retrieval import cari
from answer_processing import ambil_jawaban

# Jalankan notebook dari folder utama repository.
with Path("korpus_kalimat_final.json").open(encoding="utf-8") as file:
    kalimat_korpus = json.load(file)["kalimat"]

K = 20
print(f"Korpus siap: {len(kalimat_korpus)} kalimat | BM25 k={K}")

Korpus siap: 3140 kalimat | BM25 k=20


## 2. Pertanyaan dan jawaban

In [2]:
SOAL_UJI = [
    {
        "id_dokumen": "D01",
        "pertanyaan": "Sumber daya alam apakah yang menjadi sasaran utama Jepang dalam pendudukannya di Hindia Belanda?",
        "jenis_acuan": "apa",
        "jawaban_acuan": "Minyak bumi",
        "blok_acuan": "D01-B0009"
    },
    {
        "id_dokumen": "D02",
        "pertanyaan": "Dimana masa persidangan pertama BPUPK dibuka?",
        "jenis_acuan": "dimana",
        "jawaban_acuan": "Gedung Chuo Sangi In di Jakarta",
        "blok_acuan": "D02-B0010"
    },
    {
        "id_dokumen": "D03",
        "pertanyaan": "Siapa yang ditunjuk PPKI sebagai wakil presiden pertama Indonesia?",
        "jenis_acuan": "siapa",
        "jawaban_acuan": "Mohammad Hatta",
        "blok_acuan": "D03-B0001"
    },
    {
        "id_dokumen": "D04",
        "pertanyaan": "Siapa yang memberikan julukan \"Piagam Jakarta\" kepada naskah usulan Mukadimah Undang-Undang Dasar Indonesia?",
        "jenis_acuan": "siapa",
        "jawaban_acuan": "Mohammad Yamin",
        "blok_acuan": "D04-B0017"
    },
    {
        "id_dokumen": "D05",
        "pertanyaan": "Pada tanggal berapa Soekarno dan Mohammad Hatta dibawa ke Rengasdengklok?",
        "jenis_acuan": "kapan",
        "jawaban_acuan": "16 Agustus 1945",
        "blok_acuan": "D05-B0001"
    },
    {
        "id_dokumen": "D06",
        "pertanyaan": "Apa nama kapal tempat Jepang secara resmi menyerah kepada Sekutu?",
        "jenis_acuan": "apa",
        "jawaban_acuan": "USS Missouri",
        "blok_acuan": "D06-B0007"
    },
    {
        "id_dokumen": "D07",
        "pertanyaan": "Ke kota manakah Soekarno diasingkan bersama keluarganya pada tahun 1934?",
        "jenis_acuan": "dimana",
        "jawaban_acuan": "Ende",
        "blok_acuan": "D07-B0019"
    },
    {
        "id_dokumen": "D08",
        "pertanyaan": "Kapan Mohammad Hatta lahir?",
        "jenis_acuan": "kapan",
        "jawaban_acuan": "12 Agustus 1902",
        "blok_acuan": "D08-B0004"
    },
    {
        "id_dokumen": "D09",
        "pertanyaan": "Pada tahun berapa Sutan Sjahrir mendirikan Partai Sosialis Indonesia?",
        "jenis_acuan": "kapan",
        "jawaban_acuan": "1948",
        "blok_acuan": "D09-B0002"
    },
    {
        "id_dokumen": "D10",
        "pertanyaan": "Pada tanggal berapa Soebardjo dilantik sebagai Menteri Luar Negeri pada Kabinet Presidensial?",
        "jenis_acuan": "kapan",
        "jawaban_acuan": "18 Agustus 1945",
        "blok_acuan": "D10-B0011"
    },
    {
        "id_dokumen": "D11",
        "pertanyaan": "Dimana Fatmawati dilahirkan?",
        "jenis_acuan": "dimana",
        "jawaban_acuan": "Bengkulu",
        "blok_acuan": "D11-B0002"
    },
    {
        "id_dokumen": "D12",
        "pertanyaan": "Siapa yang terpilih menjadi panglima Tentara Keamanan Rakyat?",
        "jenis_acuan": "siapa",
        "jawaban_acuan": "Sudirman",
        "blok_acuan": "D12-B0009"
    },
    {
        "id_dokumen": "D13",
        "pertanyaan": "Siapa komandan pasukan Britania yang tewas dalam baku tembak pada 30 Oktober 1945 di Surabaya?",
        "jenis_acuan": "siapa",
        "jawaban_acuan": "Brigadir A. W. S. Mallaby",
        "blok_acuan": "D13-B0002"
    },
    {
        "id_dokumen": "D14",
        "pertanyaan": "Siapa anggota milisi BRI yang meledakkan gudang amunisi di Dayeuhkolot dengan dinamit?",
        "jenis_acuan": "siapa",
        "jawaban_acuan": "Muhammad Toha",
        "blok_acuan": "D14-B0007"
    },
    {
        "id_dokumen": "D15",
        "pertanyaan": "Di daerah mana pasukan Belanda melakukan pendaratan dalam Operasi Produk Utara?",
        "jenis_acuan": "dimana",
        "jawaban_acuan": "Pasir Poetih, Situbondo",
        "blok_acuan": "D15-B0005"
    },
    {
        "id_dokumen": "D16",
        "pertanyaan": "Operasi militer Belanda yang dikenal dengan sebutan Operasi Gagak dilancarkan pada bulan apa?",
        "jenis_acuan": "kapan",
        "jawaban_acuan": "Desember 1948",
        "blok_acuan": "D16-B0001"
    },
    {
        "id_dokumen": "D17",
        "pertanyaan": "Siapakah Letkol yang memimpin pasukan dari sektor barat hingga ke batas Malioboro dalam Serangan Umum 1 Maret 1949?",
        "jenis_acuan": "siapa",
        "jawaban_acuan": "Letkol Soeharto",
        "blok_acuan": "D17-B0004"
    },
    {
        "id_dokumen": "D18",
        "pertanyaan": "Siapa yang bertindak sebagai perantara pada tahap awal Perundingan Linggajati?",
        "jenis_acuan": "siapa",
        "jawaban_acuan": "Lord Killearn",
        "blok_acuan": "D18-B0013"
    },
    {
        "id_dokumen": "D19",
        "pertanyaan": "Dimana kapal USS Renville berlabuh saat negosiasi Perjanjian Renville berlangsung?",
        "jenis_acuan": "dimana",
        "jawaban_acuan": "Teluk Jakarta",
        "blok_acuan": "D19-B0002"
    },
    {
        "id_dokumen": "D20",
        "pertanyaan": "Siapa kepala delegasi Republik Indonesia dalam perundingan Roem-Roijen?",
        "jenis_acuan": "siapa",
        "jawaban_acuan": "Mohamad Roem",
        "blok_acuan": "D20-B0002"
    },
    {
        "id_dokumen": "D21",
        "pertanyaan": "Berapa jumlah negara bagian Republik Indonesia Serikat?",
        "jenis_acuan": "berapa",
        "jawaban_acuan": "16 negara bagian",
        "blok_acuan": "D21-B0018"
    }
]
print(f"Daftar soal siap: {len(SOAL_UJI)} pertanyaan")

Daftar soal siap: 21 pertanyaan


## 3. Pengujian seluruh pertanyaan

In [3]:
SOAL_UJI = [
    {
        "id_dokumen": "D01",
        "pertanyaan": "Sumber daya alam apakah yang menjadi sasaran utama Jepang dalam pendudukannya di Hindia Belanda?",
        "jenis_acuan": "apa",
        "jawaban_acuan": "Minyak bumi",
        "blok_acuan": "D01-B0009"
    },
    {
        "id_dokumen": "D02",
        "pertanyaan": "Dimana masa persidangan pertama BPUPK dibuka?",
        "jenis_acuan": "dimana",
        "jawaban_acuan": "Gedung Chuo Sangi In di Jakarta",
        "blok_acuan": "D02-B0010"
    },
    {
        "id_dokumen": "D03",
        "pertanyaan": "Siapa yang ditunjuk PPKI sebagai wakil presiden pertama Indonesia?",
        "jenis_acuan": "siapa",
        "jawaban_acuan": "Mohammad Hatta",
        "blok_acuan": "D03-B0001"
    },
    {
        "id_dokumen": "D04",
        "pertanyaan": "Siapa yang memberikan julukan \"Piagam Jakarta\" kepada naskah usulan Mukadimah Undang-Undang Dasar Indonesia?",
        "jenis_acuan": "siapa",
        "jawaban_acuan": "Mohammad Yamin",
        "blok_acuan": "D04-B0017"
    },
    {
        "id_dokumen": "D05",
        "pertanyaan": "Pada tanggal berapa Soekarno dan Mohammad Hatta dibawa ke Rengasdengklok?",
        "jenis_acuan": "kapan",
        "jawaban_acuan": "16 Agustus 1945",
        "blok_acuan": "D05-B0001"
    },
    {
        "id_dokumen": "D06",
        "pertanyaan": "Apa nama kapal tempat Jepang secara resmi menyerah kepada Sekutu?",
        "jenis_acuan": "apa",
        "jawaban_acuan": "USS Missouri",
        "blok_acuan": "D06-B0007"
    },
    {
        "id_dokumen": "D07",
        "pertanyaan": "Ke kota manakah Soekarno diasingkan bersama keluarganya pada tahun 1934?",
        "jenis_acuan": "dimana",
        "jawaban_acuan": "Ende",
        "blok_acuan": "D07-B0019"
    },
    {
        "id_dokumen": "D08",
        "pertanyaan": "Kapan Mohammad Hatta lahir?",
        "jenis_acuan": "kapan",
        "jawaban_acuan": "12 Agustus 1902",
        "blok_acuan": "D08-B0004"
    },
    {
        "id_dokumen": "D09",
        "pertanyaan": "Pada tahun berapa Sutan Sjahrir mendirikan Partai Sosialis Indonesia?",
        "jenis_acuan": "kapan",
        "jawaban_acuan": "1948",
        "blok_acuan": "D09-B0002"
    },
    {
        "id_dokumen": "D10",
        "pertanyaan": "Pada tanggal berapa Soebardjo dilantik sebagai Menteri Luar Negeri pada Kabinet Presidensial?",
        "jenis_acuan": "kapan",
        "jawaban_acuan": "18 Agustus 1945",
        "blok_acuan": "D10-B0011"
    },
    {
        "id_dokumen": "D11",
        "pertanyaan": "Dimana Fatmawati dilahirkan?",
        "jenis_acuan": "dimana",
        "jawaban_acuan": "Bengkulu",
        "blok_acuan": "D11-B0002"
    },
    {
        "id_dokumen": "D12",
        "pertanyaan": "Siapa yang terpilih menjadi panglima Tentara Keamanan Rakyat?",
        "jenis_acuan": "siapa",
        "jawaban_acuan": "Sudirman",
        "blok_acuan": "D12-B0009"
    },
    {
        "id_dokumen": "D13",
        "pertanyaan": "Siapa komandan pasukan Britania yang tewas dalam baku tembak pada 30 Oktober 1945 di Surabaya?",
        "jenis_acuan": "siapa",
        "jawaban_acuan": "Brigadir A. W. S. Mallaby",
        "blok_acuan": "D13-B0002"
    },
    {
        "id_dokumen": "D14",
        "pertanyaan": "Siapa anggota milisi BRI yang meledakkan gudang amunisi di Dayeuhkolot dengan dinamit?",
        "jenis_acuan": "siapa",
        "jawaban_acuan": "Muhammad Toha",
        "blok_acuan": "D14-B0007"
    },
    {
        "id_dokumen": "D15",
        "pertanyaan": "Di daerah mana pasukan Belanda melakukan pendaratan dalam Operasi Produk Utara?",
        "jenis_acuan": "dimana",
        "jawaban_acuan": "Pasir Poetih, Situbondo",
        "blok_acuan": "D15-B0005"
    },
    {
        "id_dokumen": "D16",
        "pertanyaan": "Operasi militer Belanda yang dikenal dengan sebutan Operasi Gagak dilancarkan pada bulan apa?",
        "jenis_acuan": "kapan",
        "jawaban_acuan": "Desember 1948",
        "blok_acuan": "D16-B0001"
    },
    {
        "id_dokumen": "D17",
        "pertanyaan": "Siapakah Letkol yang memimpin pasukan dari sektor barat hingga ke batas Malioboro dalam Serangan Umum 1 Maret 1949?",
        "jenis_acuan": "siapa",
        "jawaban_acuan": "Letkol Soeharto",
        "blok_acuan": "D17-B0004"
    },
    {
        "id_dokumen": "D18",
        "pertanyaan": "Siapa yang bertindak sebagai perantara pada tahap awal Perundingan Linggajati?",
        "jenis_acuan": "siapa",
        "jawaban_acuan": "Lord Killearn",
        "blok_acuan": "D18-B0013"
    },
    {
        "id_dokumen": "D19",
        "pertanyaan": "Dimana kapal USS Renville berlabuh saat negosiasi Perjanjian Renville berlangsung?",
        "jenis_acuan": "dimana",
        "jawaban_acuan": "Teluk Jakarta",
        "blok_acuan": "D19-B0002"
    },
    {
        "id_dokumen": "D20",
        "pertanyaan": "Siapa kepala delegasi Republik Indonesia dalam perundingan Roem-Roijen?",
        "jenis_acuan": "siapa",
        "jawaban_acuan": "Mohamad Roem",
        "blok_acuan": "D20-B0002"
    },
    {
        "id_dokumen": "D21",
        "pertanyaan": "Berapa jumlah negara bagian Republik Indonesia Serikat?",
        "jenis_acuan": "berapa",
        "jawaban_acuan": "16 negara bagian",
        "blok_acuan": "D21-B0018"
    }
]

## 3. Fungsi chatbot terintegrasi

In [4]:
def jawab_pertanyaan(pertanyaan, k=K):
    informasi = process_question(pertanyaan)
    kandidat = cari(pertanyaan, k=k)
    hasil = ambil_jawaban(
        informasi,
        kandidat,
        kalimat_korpus=kalimat_korpus,
    ) if kandidat else None
    return {
        "info": informasi,
        "jawaban": hasil["jawaban"] if hasil else None,
        "id_sumber": hasil["id_kalimat"] if hasil else None,
        "kandidat": kandidat,
    }

## 4. Pengujian soal

In [5]:
hasil_akhir = []

for soal in SOAL_UJI:
    hasil = jawab_pertanyaan(soal["pertanyaan"])
    jawaban = hasil["jawaban"]
    cocok = (
        jawaban is not None
        and str(jawaban).strip().casefold()
        == soal["jawaban_acuan"].strip().casefold()
    )

    hasil_akhir.append({
        "id": soal["id_dokumen"],
        "pertanyaan": soal["pertanyaan"],
        "tipe_jawaban": hasil["info"]["answer_type"],
        "jawaban_acuan": soal["jawaban_acuan"],
        "jawaban_chatbot": jawaban,
        "id_sumber": hasil["id_sumber"],
        "cocok_persis": cocok,
    })
    print(f'{soal["id_dokumen"]} | {"COCOK" if cocok else "PERLU CEK"} | {jawaban}')

D01 | COCOK | Minyak bumi
D02 | COCOK | Gedung Chuo Sangi In di Jakarta
D03 | COCOK | Mohammad Hatta
D04 | COCOK | Mohammad Yamin
D05 | COCOK | 16 Agustus 1945
D06 | COCOK | USS Missouri
D07 | COCOK | Ende
D08 | COCOK | 12 Agustus 1902
D09 | COCOK | 1948
D10 | COCOK | 18 Agustus 1945
D11 | COCOK | Bengkulu
D12 | COCOK | Sudirman
D13 | COCOK | Brigadir A. W. S. Mallaby
D14 | COCOK | Muhammad Toha
D15 | COCOK | Pasir Poetih, Situbondo
D16 | COCOK | Desember 1948
D17 | COCOK | Letkol Soeharto
D18 | COCOK | Lord Killearn
D19 | COCOK | Teluk Jakarta
D20 | COCOK | Mohamad Roem
D21 | COCOK | 16 negara bagian


## 5. Ringkasan dan simpan hasil

In [6]:
jumlah_soal = len(hasil_akhir)
jumlah_cocok = sum(baris["cocok_persis"] for baris in hasil_akhir)
akurasi = 100 * jumlah_cocok / jumlah_soal if jumlah_soal else 0

print(f"\nHASIL AKHIR (BM25 k={K})")
print(f"Cocok persis: {jumlah_cocok}/{jumlah_soal} ({akurasi:.2f}%)")

file_json = Path("hasil_jawaban_chatbot_21_k20.json")
file_csv = Path("hasil_jawaban_chatbot_21_k20.csv")

file_json.write_text(
    json.dumps(hasil_akhir, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
with file_csv.open("w", encoding="utf-8-sig", newline="") as file:
    penulis = csv.DictWriter(file, fieldnames=list(hasil_akhir[0]))
    penulis.writeheader()
    penulis.writerows(hasil_akhir)

print(f"File evaluasi: {file_json} dan {file_csv}")


HASIL AKHIR (BM25 k=20)
Cocok persis: 21/21 (100.00%)
File evaluasi: hasil_jawaban_chatbot_21_k20.json dan hasil_jawaban_chatbot_21_k20.csv
